# 04 — Bias Mitigation

**FairLens AI / NyayaLens — Person 3 (ML Layer)**

This notebook applies **Fairlearn ThresholdOptimizer** to produce fairer
predictions and compares before/after fairness metrics.

### What we do here
1. Rebuild data, model, and baseline bias metrics
2. Apply ThresholdOptimizer (post-processing, equalized_odds constraint)
3. Evaluate mitigated predictions
4. Compute mitigated fairness metrics
5. Print a before/after comparison table
6. Visualize the comparison
7. Save results to JSON

In [ ]:
import json
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Fairlearn MetricFrame: per-group metric computation
from fairlearn.metrics import MetricFrame

# ThresholdOptimizer: post-processing mitigation that finds per-group
# decision thresholds satisfying a fairness constraint.
from fairlearn.postprocessing import ThresholdOptimizer

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

## Step 1 — Rebuild data, model, and baseline metrics

In [ ]:
# --- Load & preprocess (same as Notebook 02/03) ---
raw_df = fetch_openml("adult", version=2, as_frame=True).frame
cleaned_df = raw_df.copy()
for col in cleaned_df.columns:
    if cleaned_df[col].dtype == object:
        cleaned_df = cleaned_df[cleaned_df[col] != "?"]
cleaned_df = cleaned_df.dropna()

income_labels = cleaned_df["class"].apply(
    lambda v: 1 if ">50K" in str(v) else 0
)
gender_sensitive = cleaned_df["sex"].copy()
feature_df = cleaned_df.drop(columns=["sex", "race", "fnlwgt", "class"])
feature_df = pd.get_dummies(feature_df, drop_first=True)

features_train, features_test, labels_train, labels_test = train_test_split(
    feature_df, income_labels, test_size=0.2, random_state=42
)

scaler = StandardScaler()
features_train = pd.DataFrame(
    scaler.fit_transform(features_train),
    columns=features_train.columns, index=features_train.index
)
features_test = pd.DataFrame(
    scaler.transform(features_test),
    columns=features_test.columns, index=features_test.index
)

gender_train = gender_sensitive.loc[features_train.index]
gender_test = gender_sensitive.loc[features_test.index]

# --- Train baseline ---
baseline_model = LogisticRegression(max_iter=5000, random_state=42)
baseline_model.fit(features_train, labels_train)
baseline_predictions = baseline_model.predict(features_test)

# --- Baseline metrics ---
baseline_metrics = {
    "accuracy":  float(accuracy_score(labels_test, baseline_predictions)),
    "precision": float(precision_score(labels_test, baseline_predictions)),
    "recall":    float(recall_score(labels_test, baseline_predictions)),
    "f1":        float(f1_score(labels_test, baseline_predictions)),
}
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")

# --- Baseline fairness ---
baseline_fairness = MetricFrame(
    metrics={"accuracy": accuracy_score, "recall": recall_score},
    y_true=labels_test, y_pred=baseline_predictions,
    sensitive_features=gender_test,
)
print("\nBaseline per-group:")
print(baseline_fairness.by_group)
print(f"\nBaseline differences: {baseline_fairness.difference().to_dict()}")

## Step 2 — Apply ThresholdOptimizer

**What is ThresholdOptimizer?**

It's a **post-processing** fairness method from Fairlearn. It works like this:

1. Takes your already-trained model (doesn't retrain it)
2. Uses the model's `predict_proba()` to get probability scores
3. Finds a **separate decision threshold** for each group (Male, Female)
4. Chooses thresholds that satisfy the fairness constraint (equalized_odds)
5. Returns adjusted predictions

**Why equalized_odds?**
Equalized odds requires that both True Positive Rate (recall) AND
False Positive Rate are equal across groups. This directly targets
the recall gap we detected.

**Why not Reweighing (AIF360)?**
We tested Reweighing first. It adjusts training sample weights, but
Logistic Regression on the Adult dataset absorbs those weights without
changing decision boundaries enough. ThresholdOptimizer acts *after*
training and directly controls per-group thresholds — much more effective.

In [ ]:
# ThresholdOptimizer wraps our baseline model.
# - constraints="equalized_odds": enforce equal TPR and FPR across groups
# - objective="balanced_accuracy_score": optimise for balanced accuracy
# - predict_method="predict_proba": use probability scores for smooth thresholds
mitigated_predictor = ThresholdOptimizer(
    estimator=baseline_model,
    constraints="equalized_odds",
    objective="balanced_accuracy_score",
    predict_method="predict_proba",
)

# .fit() learns the per-group thresholds from the training data
mitigated_predictor.fit(
    features_train,
    labels_train,
    sensitive_features=gender_train,
)

# .predict() applies the learned thresholds to produce fair predictions
mitigated_predictions = mitigated_predictor.predict(
    features_test,
    sensitive_features=gender_test,
)

print("ThresholdOptimizer mitigation applied.")
print(f"Unique mitigated prediction values: {np.unique(mitigated_predictions)}")

## Step 3 — Evaluate mitigated predictions

In [ ]:
mitigated_metrics = {
    "accuracy":  float(accuracy_score(labels_test, mitigated_predictions)),
    "precision": float(precision_score(labels_test, mitigated_predictions)),
    "recall":    float(recall_score(labels_test, mitigated_predictions)),
    "f1":        float(f1_score(labels_test, mitigated_predictions)),
}
print("Mitigated metrics:")
for k, v in mitigated_metrics.items():
    print(f"  {k}: {v:.4f}")

## Step 4 — Mitigated fairness metrics

In [ ]:
mitigated_fairness = MetricFrame(
    metrics={"accuracy": accuracy_score, "recall": recall_score},
    y_true=labels_test, y_pred=mitigated_predictions,
    sensitive_features=gender_test,
)
print("Mitigated per-group:")
print(mitigated_fairness.by_group)
print(f"\nMitigated differences: {mitigated_fairness.difference().to_dict()}")

## Step 5 — Before/After comparison table

In [ ]:
before_groups = baseline_fairness.by_group
after_groups = mitigated_fairness.by_group

print(f"{'Metric':<25} {'Before':>10} {'After':>10}")
print("-" * 47)
print(f"{'Accuracy':<25} {baseline_metrics['accuracy']:>10.4f} {mitigated_metrics['accuracy']:>10.4f}")
print(f"{'Precision':<25} {baseline_metrics['precision']:>10.4f} {mitigated_metrics['precision']:>10.4f}")
print(f"{'Recall':<25} {baseline_metrics['recall']:>10.4f} {mitigated_metrics['recall']:>10.4f}")
print(f"{'F1':<25} {baseline_metrics['f1']:>10.4f} {mitigated_metrics['f1']:>10.4f}")
print()

for group in before_groups.index:
    br = before_groups.loc[group, 'recall']
    ar = after_groups.loc[group, 'recall']
    print(f"{'Recall (' + str(group) + ')':<25} {br:>10.4f} {ar:>10.4f}")

before_diff = baseline_fairness.difference()['recall']
after_diff = mitigated_fairness.difference()['recall']
print(f"{'Recall Difference':<25} {before_diff:>10.4f} {after_diff:>10.4f}")

## Step 6 — Visualize the comparison

In [ ]:
groups = list(before_groups.index)
x = np.arange(len(groups))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy comparison ---
before_acc = [before_groups.loc[g, 'accuracy'] for g in groups]
after_acc = [after_groups.loc[g, 'accuracy'] for g in groups]
axes[0].bar(x - width/2, before_acc, width, label='Before', color='#F44336', alpha=0.8)
axes[0].bar(x + width/2, after_acc, width, label='After', color='#4CAF50', alpha=0.8)
axes[0].set_title('Accuracy by Group: Before vs After')
axes[0].set_ylabel('Accuracy')
axes[0].set_xticks(x)
axes[0].set_xticklabels(groups)
axes[0].set_ylim(0.5, 1.0)
axes[0].legend()

# --- Recall comparison ---
before_rec = [before_groups.loc[g, 'recall'] for g in groups]
after_rec = [after_groups.loc[g, 'recall'] for g in groups]
axes[1].bar(x - width/2, before_rec, width, label='Before', color='#F44336', alpha=0.8)
axes[1].bar(x + width/2, after_rec, width, label='After', color='#4CAF50', alpha=0.8)
axes[1].set_title('Recall by Group: Before vs After')
axes[1].set_ylabel('Recall')
axes[1].set_xticks(x)
axes[1].set_xticklabels(groups)
axes[1].set_ylim(0.0, 1.0)
axes[1].legend()

plt.suptitle('Bias Mitigation: ThresholdOptimizer (equalized_odds)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Recall difference bar chart ---
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Before', 'After'], [before_diff, after_diff],
       color=['#F44336', '#4CAF50'], alpha=0.8)
ax.set_ylabel('Recall Difference (max gap)')
ax.set_title('Recall Disparity: Before vs After Mitigation')
ax.axhline(y=0.10, color='orange', linestyle='--', label='HIGH threshold (10%)')
ax.axhline(y=0.05, color='gray', linestyle='--', label='MEDIUM threshold (5%)')
ax.legend()
plt.tight_layout()
plt.show()

## Step 7 — Save results to JSON

This produces the same structured output that Person 2's API endpoint
will eventually return.

In [ ]:
# Build structured output
before_bias = {
    "by_group": {},
    "differences": {
        "accuracy_difference": float(round(baseline_fairness.difference()['accuracy'], 4)),
        "recall_difference":   float(round(baseline_fairness.difference()['recall'], 4)),
    }
}
for g in before_groups.index:
    before_bias["by_group"][str(g)] = {
        "accuracy": float(before_groups.loc[g, 'accuracy']),
        "recall":   float(before_groups.loc[g, 'recall']),
    }

after_bias = {
    "by_group": {},
    "differences": {
        "accuracy_difference": float(round(mitigated_fairness.difference()['accuracy'], 4)),
        "recall_difference":   float(round(mitigated_fairness.difference()['recall'], 4)),
    }
}
for g in after_groups.index:
    after_bias["by_group"][str(g)] = {
        "accuracy": float(after_groups.loc[g, 'accuracy']),
        "recall":   float(after_groups.loc[g, 'recall']),
    }

accuracy_change = mitigated_metrics['accuracy'] - baseline_metrics['accuracy']
recall_diff_change = after_diff - before_diff

final_output = {
    "status": "success",
    "dataset": "UCI Adult (Census Income)",
    "sensitive_column": "sex",
    "target_column": "income (class)",
    "mitigation_method": "Fairlearn ThresholdOptimizer (equalized_odds)",
    "before": {"model_metrics": baseline_metrics, "bias_metrics": before_bias},
    "after":  {"model_metrics": mitigated_metrics, "bias_metrics": after_bias},
    "improvement": {
        "accuracy_change": float(round(accuracy_change, 4)),
        "recall_difference_change": float(round(recall_diff_change, 4)),
    }
}

# Save to outputs
output_path = os.path.join("..", "outputs", "reports", "adult_audit.json")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w") as f:
    json.dump(final_output, f, indent=2)

print(f"Results saved to: {output_path}")
print()
print(json.dumps(final_output, indent=2))

## Summary

- **ThresholdOptimizer** uses post-processing to find per-group decision thresholds
  that satisfy equalized odds.
- The recall gap between Male and Female is significantly reduced.
- There may be a small accuracy trade-off — this is expected and normal.
- The JSON output matches the contract Person 2 expects from the API.

**Next:** Notebook 05 — Counterfactual Checks